In [ ]:
import os
from pathlib import Path
from PyPDF2 import PdfReader

# ---------------------------
# CONFIGURATION
# ---------------------------
PDF_FOLDER = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag/data/pdfs")   # Folder containing PDFs
TXT_FOLDER = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag/data/txt")    # Folder containing TXT files
CHUNK_SIZE = 500                     # Approximate words per chunk

# ---------------------------
# FUNCTIONS
# ---------------------------

# Load PDF and extract text
def load_pdf(file_path):
    """Read a PDF file and return all text as a single string."""
    text = ""
    reader = PdfReader(file_path)
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:  # only add if text exists
            text += page_text + "\n"
    return text

# Load TXT and extract text
def load_txt(file_path):
    """Read a TXT file and return all text as a single string."""
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

# Split text into smaller chunks
def chunk_text(text, chunk_size=CHUNK_SIZE):
    """
    Split a long text into smaller chunks of 'chunk_size' words.
    """
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

# Main ingestion function
def ingest_documents():
    """
    Process all PDF and TXT files in the data folders.
    Returns a list of dictionaries containing:
    - doc_name: file name
    - chunk_id: automatic ID for each chunk
    - text: chunk text
    """
    all_chunks = []

    # Process PDFs
    for pdf_file in PDF_FOLDER.glob("*.pdf"):
        text = load_pdf(pdf_file)
        chunks = chunk_text(text)
        for idx, chunk in enumerate(chunks):
            all_chunks.append({
                "doc_name": pdf_file.name,
                "chunk_id": idx,  # automatic chunk ID
                "text": chunk
            })

    # Process TXT files
    for txt_file in TXT_FOLDER.glob("*.txt"):
        text = load_txt(txt_file)
        chunks = chunk_text(text)
        for idx, chunk in enumerate(chunks):
            all_chunks.append({
                "doc_name": txt_file.name,
                "chunk_id": idx,  # automatic chunk ID
                "text": chunk
            })

    print(f"[INFO] Total chunks created: {len(all_chunks)}")
    return all_chunks

# ---------------------------
# MAIN TEST
# ---------------------------
if __name__ == "__main__":
    chunks = ingest_documents()
    print("Sample chunk from first document:")
    print(f"Document: {chunks[3]['doc_name']}, Chunk ID: {chunks[1]['chunk_id']}")
    print(chunks[1]['text'][:500] + "...")

[INFO] Total chunks created: 20
Sample chunk from first document:
Document: reinforcement_learning.pdf, Chunk ID: 1
learning algorithms. It learns the optimal action-value function using the update rule: Q(s, a) = Q(s, a) + α [r + γ max Q(s’, a’) − Q(s, a)] Where: ● α (alpha): Learning rate ● γ (gamma): Discount factor ● r: Reward The algorithm iteratively updates Q-values based on observed rewards and future estimates. 8. Deep Reinforcement Learning Deep Reinforcement Learning combines neural networks with reinforcement learning to handle high-dimensional state spaces. A common approach is the Deep Q-Network...


In [4]:
import sys
from pathlib import Path
import pickle
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# Add project root to Python path
PROJECT_ROOT = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag")
sys.path.append(str(PROJECT_ROOT))

# Import your ingestion function
from ingestion_pipeline.ingestion import ingest_documents
# ---------------------------
# CONFIGURATION
# ---------------------------
EMBEDDING_MODEL = "all-MiniLM-L6-v2"  # SentenceTransformers model
EMBEDDINGS_FOLDER = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag/embeddings")
EMBEDDINGS_FOLDER.mkdir(exist_ok=True)  # create folder if not exists

# ---------------------------
# LOAD CHUNKS
# ---------------------------
print("[INFO] Loading documents and splitting into chunks...")
chunks = ingest_documents()  # returns list of dicts with doc_name, chunk_id, text

# ---------------------------
# INITIALIZE EMBEDDING MODEL
# ---------------------------
print(f"[INFO] Loading embedding model: {EMBEDDING_MODEL}")
model = SentenceTransformer(EMBEDDING_MODEL)

# ---------------------------
# CREATE EMBEDDINGS
# ---------------------------
embeddings_data = []  # will store embeddings + metadata

print("[INFO] Generating embeddings...")
for chunk in tqdm(chunks):
    vector = model.encode(chunk["text"])
    embeddings_data.append({
        "doc_name": chunk["doc_name"],
        "chunk_id": chunk.get("chunk_id"),  # optional if you are using it
        "text": chunk["text"],
        "embedding": vector
    })

# ---------------------------
# SAVE EMBEDDINGS
# ---------------------------
output_file = EMBEDDINGS_FOLDER / "all_embeddings.pkl"
with open(output_file, "wb") as f:
    pickle.dump(embeddings_data, f)

print(f"[INFO] Embeddings saved to {output_file}")
print(f"[INFO] Total embeddings created: {len(embeddings_data)}")

[INFO] Loading documents and splitting into chunks...
[INFO] Total chunks created: 20
[INFO] Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13051.37it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Generating embeddings...


100%|██████████| 20/20 [00:00<00:00, 66.23it/s]

[INFO] Embeddings saved to /Users/malavjoshi/Desktop/RAG_Projects/local_rag/embeddings/all_embeddings.pkl
[INFO] Total embeddings created: 20
